<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/01_Data_Scientist/intermediate/04_model_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Evaluation — Metrics Deep Dive

Choosing the wrong metric is one of the most dangerous mistakes in DS. This notebook covers every evaluation metric you need, when to use each, and how to build a complete evaluation dashboard.

**Topics:** Classification metrics, regression metrics, calibration, learning curves, custom scorers

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score, confusion_matrix,
                              roc_curve, precision_recall_curve, classification_report,
                              mean_absolute_error, mean_squared_error, r2_score)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

# Imbalanced classification dataset (10% positive)
X, y = make_classification(n_samples=10000, n_features=20, n_informative=10,
                            weights=[0.9, 0.1], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Dataset: {len(X_train)} train, {len(X_test)} test')
print(f'Class distribution: {np.bincount(y_test)} (positive rate: {y_test.mean():.1%})')

## 1. The Accuracy Paradox

In [ ]:
# With 10% positive class, a naive "predict always negative" classifier gets 90% accuracy!
y_naive = np.zeros(len(y_test))  # always predict class 0
y_smart = GradientBoostingClassifier(random_state=42).fit(X_train, y_train).predict(X_test)
probs_smart = GradientBoostingClassifier(random_state=42).fit(X_train, y_train).predict_proba(X_test)[:, 1]

print('Comparing naive vs trained model:')
print(f'{"Metric":<25} {"Naive (all 0)":>15} {"Trained GBM":>15}')
print('-'*60)
metrics = [
    ('Accuracy', accuracy_score(y_test, y_naive), accuracy_score(y_test, y_smart)),
    ('Precision', precision_score(y_test, y_naive, zero_division=0), precision_score(y_test, y_smart)),
    ('Recall', recall_score(y_test, y_naive), recall_score(y_test, y_smart)),
    ('F1 Score', f1_score(y_test, y_naive), f1_score(y_test, y_smart)),
    ('AUC-ROC', 0.5, roc_auc_score(y_test, probs_smart)),
    ('Avg Precision', average_precision_score(y_test, y_naive), average_precision_score(y_test, probs_smart)),
]
for name, naive_val, smart_val in metrics:
    print(f'{name:<25} {naive_val:>15.4f} {smart_val:>15.4f}')

print('\n→ Accuracy looks good for naive model (90%)!')
print('→ F1, Recall, AUC expose the truth.')

## 2. Confusion Matrix & Threshold Selection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_smart)
ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]}\n({cm[i,j]/len(y_test):.1%})', 
                ha='center', va='center', fontsize=12, fontweight='bold')
ax.set_xticks([0,1]); ax.set_yticks([0,1])
ax.set_xticklabels(['Pred 0', 'Pred 1']); ax.set_yticklabels(['True 0', 'True 1'])
ax.set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, thresholds_roc = roc_curve(y_test, probs_smart)
roc_auc = roc_auc_score(y_test, probs_smart)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={roc_auc:.3f}')
axes[1].plot([0,1],[0,1],'k--', label='Random')
axes[1].set_xlabel('FPR (1 - Specificity)'); axes[1].set_ylabel('TPR (Recall)')
axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Precision-Recall Curve (better for imbalanced data!)
prec, rec, thresholds_pr = precision_recall_curve(y_test, probs_smart)
ap = average_precision_score(y_test, probs_smart)
axes[2].plot(rec, prec, 'r-', lw=2, label=f'AP={ap:.3f}')
axes[2].axhline(y_test.mean(), color='k', ls='--', label=f'Baseline ({y_test.mean():.3f})')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve\n(better for imbalanced data!)')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# Threshold optimization
print('\nPrecision-Recall tradeoff at different thresholds:')
print(f'{"Threshold":<12} {"Precision":<12} {"Recall":<12} {"F1":<12}')
for thresh in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_thresh = (probs_smart >= thresh).astype(int)
    p = precision_score(y_test, y_pred_thresh, zero_division=0)
    r = recall_score(y_test, y_pred_thresh)
    f1 = f1_score(y_test, y_pred_thresh)
    print(f'{thresh:<12.1f} {p:<12.3f} {r:<12.3f} {f1:<12.3f}')

## 3. Probability Calibration

A well-calibrated model means: when it says 70% probability, the true rate IS 70%. This is critical for risk scoring, medical diagnosis, fraud detection.

In [ ]:
from sklearn.calibration import CalibrationDisplay

models_to_calibrate = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GBM': GradientBoostingClassifier(random_state=42),
    'Logistic Regression': Pipeline([('s', StandardScaler()), 
                                      ('clf', LogisticRegression(max_iter=1000))]),
}

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0,1],[0,1],'k--', label='Perfectly calibrated')
colors = ['steelblue', 'darkorange', 'green']

for (name, model), color in zip(models_to_calibrate.items(), colors):
    model.fit(X_train, y_train)
    probs = model.predict_proba(X_test)[:, 1]
    fraction_pos, mean_pred = calibration_curve(y_test, probs, n_bins=10)
    ax.plot(mean_pred, fraction_pos, 's-', color=color, lw=2, label=name)

ax.set_xlabel('Mean Predicted Probability'); ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curves\n(closer to diagonal = better calibrated)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('Calibration matters when probabilities are used directly (not just rank order).')
print('Fix poor calibration with: CalibratedClassifierCV(model, method="isotonic" or "sigmoid")')

## Metric Selection Guide

| Scenario | Primary Metric | Secondary |
|----------|---------------|----------|
| Balanced classes | Accuracy, F1 | AUC-ROC |
| Imbalanced classes | F1 (weighted), AUC-PR | Recall or Precision |
| High false positive cost (spam filter) | Precision | F1 |
| High false negative cost (cancer, fraud) | Recall | F1 |
| Probability needed | AUC-ROC, Log-loss | Calibration curve |
| Regression | RMSE (sensitive to outliers) | MAE, MAPE, R² |
| Ranking problem | NDCG, MAP | MRR |